In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pagerank_implementation import * 

In [ ]:
def power_iteration_custom_vector(m_prime, v_initial, epsilon = 1e-6, max_iterations = 1000):
    x = v_initial.copy() # Just change the uniform distribution to personalized vector -> The rest is the same
    error_history = [] # Storage for convergence error tracking
    
    # Main iteration loop
    for iteration in range(max_iterations):
        x_new = m_prime @ x
        
        error = np.linalg.norm(x_new - x, ord=1)
        error_history.append(error)
        
        if error < epsilon: 
            print(f"Converged in {iteration + 1} with Final error = {error:.2e}\n")
            return x_new, error_history
        
        x = x_new # update vector
    
    print (f"Did not converge in {max_iterations} iterations")
    
    return x, error_history

In [ ]:
#Define Personal Transition Matrix with Damping Effect 
def personal_transition_matrix (transition_matrix, personalization_vector, damping_factor = 0.85):
    n = transition_matrix.shape[0] #erturns the number of pages
    d = damping_factor

    personalization_matrix = np.outer(personalization_vector,np.ones(n)) #v_personalized * 1^T
    m_prime = d * transition_matrix + (1 - d) * personalization_matrix # apply modified scoring formula

    return m_prime

In [ ]:
def find_ev_modified_remove (A_original, links_to_remove, damping_factor = 0.85):
    a_mod = A_original.copy()

    for (destination, origin) in links_to_remove:
        a_mod[origin,destination] = 0

    m_mod = build_transition_matrix(a_mod)
    m_prime_mod = add_damping(m_mod)
    r_mod = find_ev(m_prime_mod)

    return a_mod, r_mod

In [ ]:
def find_ev_modified_add (A_original, links_to_add, damping_factor = 0.85):
    a_mod = A_original.copy()

    for (destination, origin) in links_to_add:
        a_mod[origin,destination] = 1

    m_mod = build_transition_matrix(a_mod)
    m_prime_mod = add_damping(m_mod,damping_factor)
    r_mod = find_ev(m_prime_mod)

    return a_mod, r_mod

In [ ]:
d = 0.85 #damping_factor

n = m_prime_web.shape[0] #erturns the number of pages in the web
v_uniform = np.ones(n) / n # the uniform distribution vector - default vector used in Section 3 

v_pageA = np.zeros(n) # personalized vector with 1x8 (n) zeros
v_pageA[0] = 1 # all probability goes to page A - we visit only page A

In [ ]:
r_uni, history_uni = power_iteration_custom_vector(m_prime_web, v_uniform)
r_pageA, history_pageA = power_iteration_custom_vector(m_prime_web, v_pageA)

In [ ]:
print("Side-by-Side Comparison:")
print("-"*65)
print("Page | Uniform V  |  PageA V  |    EV    | Uni - EV | PageA - EV")
print("-"*65)
for i, label in enumerate(page_labels):
    diff_1 = abs(r_uni[i] - r_eig[i])
    diff_2 = abs(r_pageA[i] - r_eig[i])
    print(f"  {label}  |  {r_uni[i]:.6f}  |  {r_pageA[i]:.6f} | {r_pageA[i]:.6f} | {diff_1:.6f} | {diff_2:.6f}")

In [ ]:
all_results = {
    'Uniform': (r_uni, history_uni),
    'Biased A': (r_pageA, history_pageA),
    'EigenValues': (r_eig,error)
} 

num_random = 3 # generate 3 random initial vectors

for i in range(num_random):
    alpha = np.ones(n)
    v_random = np.random.dirichlet(alpha)

    r_random, history_random = power_iteration_custom_vector(m_prime_web, v_random)

    label = f'Random_{i+1}'
    all_results[label] = (r_random, history_random)

In [ ]:
keys = list(all_results.keys())

print("P | EigenValues| Uniform   | Biased A  | Random V1 | Random V2 | Random V3  ")
print("-" * 80)

for i, label in enumerate(page_labels):
    row_values = [all_results[k][0][i] for k in keys]
    formatted_scores = " |".join(f"{val:10.6f}" for val in row_values)
    print(f"{label} | {formatted_scores}" )

In [ ]:
m_prime_pageA = personal_transition_matrix(m_prime_web,v_pageA)

print(f"Transition Matrix with biased behaviour\n")
print (f"{m_prime_pageA}\n")
print(f"Sum of Columns: {m_prime_pageA.sum(axis=0)} (should be 1.0 for all columns)")

In [ ]:
r_eig, error #Standard Eigenvector page scores
r_ev_pageA = find_ev(m_prime_pageA) # apply the personalized transition matrix

print("\nPage | Standard | Biased A  | Change_ev  ")
print("-"*50)
for i, label in enumerate(page_labels):
    change_ev = r_ev_pageA[i] - r_eig[i]
    print(f"  {label}  | {r_eig[i]:.6f} | {r_ev_pageA[i]:.6f} | {change_ev:+.6f}")

In [ ]:
#Test with Spider Trap Personalization
main_cluster = [0,1,2,3,4] #Main Pages A,B,C,D,E
spider_trap = [5,6] #F-G spider Trap

v_trap = np.zeros(n)

for i in spider_trap:
    v_trap[i] = 0.5 # Assing 50/50 probability to the pages in the spider trap F-G

m_prime_trap = personal_transition_matrix(m_web,v_trap)

r_trap = find_ev(m_prime_trap) 

sorted_indices_trap = np.argsort(r_trap)[::-1]  # Descending order
for rank, idx in enumerate(sorted_indices_trap, 1):
    label = page_labels[idx]
    score = r_trap[idx]
    print(f"  Rank {rank}: Page {label}  -  {score:.6f}  ({score * 100:.2f}%)")

    #Standard vs Pers. A vs Pers. F+G
print("Page | Standard | Pers. A  | Trap F+G  |")
print("-"*40)
for i, label in enumerate(page_labels):  
    print(f"  {label}  | {r_eig[i]:.6f} | {r_ev_pageA[i]:.6f} | {r_trap[i]:.6f} | ")

In [ ]:
#Remove the link from C to A
links_to_remove_CA = [(2,0)]
a_mod_CA, r_mod_CA =  find_ev_modified_remove(A_web, links_to_remove_CA)

#Remove the link between D and E
links_to_remove_DE = [(3,4)]
a_mod_DE, r_mod_DE = find_ev_modified_remove(A_web,links_to_remove_DE)

# Remove the link from E to F
links_to_remove_EF = [(4,5)]
a_mod_EF, r_mod_EF = find_ev_modified_remove(A_web,links_to_remove_EF)

# Remove all three links simultaneously D→E, C→A, E→F
links_to_remove_all = [(3, 4), (2, 0), (4, 5)]  
a_remove_all, r_remove_all = find_ev_modified_remove (A_web, links_to_remove_all)

In [ ]:
all_results_remove = {
    'Standard': r_eig,
    'D->E': r_mod_DE,
    'C->A': r_mod_CA,
    'E->F': r_mod_EF,
    'Remove All': r_remove_all
}

fig = plt.plot(figsize=(15, 6))
    
# Plot 1: PageRank scores comparison
rank_values = sorted(all_results_remove.keys())
x = np.arange(len(page_labels))
width = 0.8 / len(rank_values)
    
for i, d in enumerate(rank_values):
    r = all_results_remove[d]
    offset = (i - len(rank_values)/2) * width + width/2
    plt.bar(x + offset, r, width, label=f'{d}', alpha=0.8)
    
plt.xlabel('Page', fontsize=12, fontweight='bold')
plt.ylabel('PageRank Score', fontsize=12, fontweight='bold')
plt.title('PageRank Scores in Network Modifications', fontsize=14, fontweight='bold')
plt.xticks(x, page_labels)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()


print("Page |  Standard Web  |  Remove D->E  |  Remove C-> A |  Remove E-> F |  Remove all")
print("-"*85)
for i, label in enumerate(page_labels):
    row_values = [all_results_remove[k][i] for k in all_results_remove.keys()]
    formatted_scores = " |".join(f"{val:14.6f}" for val in row_values)
    print(f"{label:<4} | {formatted_scores}" )

In [ ]:
#Experiment 1 - Add link from page F to page H
links_to_add_FH = [(5,7)]
a_modFH, r_mod_FH =  find_ev_modified_add(A_web, links_to_add_FH) 

changes_FH = np.abs(r_mod_FH - r_eig) # Full Ranking Comparison

print (f"\nMost affected pages - Descending")
print (f"Page |Rank |   Old Score   |  New Score  | Change %")
print ("-"*50)
sorted_changes_mod = np.argsort(changes_FH)[::-1]  # Descending order
for i in (sorted_changes_mod):
    label = page_labels[i]
    old_score = r_eig[i]
    new_score = r_mod_FH[i]
    change = changes_FH[i]
    print(f" {label:^3} |{i+1:^5}|{old_score:^15.6f}|{new_score:^13.6f}|{change*100:^6.2f}%")
print(f"\n")


#Experiment 2: add many links to a single page
links_to_add_many = ([0,7],[1,7],[2,7])
a_mod_many, r_mod_many =  find_ev_modified_add(A_web, links_to_add_many) 

changes_many = np.abs(r_mod_many - r_eig) # Full Ranking Comparison

print (f"\nMost affected pages - Descending")
print (f"Page |Rank |   Old Score   |  New Score  | Change %")
print ("-"*50)
sorted_changes_mod = np.argsort(changes_many)[::-1]  # Descending order
for i in (sorted_changes_mod):
    label = page_labels[i]
    old_score = r_eig[i]
    new_score = r_mod_many[i]
    change = changes_many[i]
    print(f" {label:^3} |{i+1:^5}|{old_score:^15.6f}|{new_score:^13.6f}|{change*100:^6.2f}% ")


In [ ]:
#Experiment 3: Test Damping Effect influence on link manipulation
damping_effects = [0.25, 0.50, 0.85]
results_de = {}

links_to_add = ([0,7],[1,7],[2,7])

for d in damping_effects:
    a_mod_d,r_mod_d = find_ev_modified_add(A_web, links_to_add, d)
    results_de[d] = [r_mod_d,a_mod_d]

fig = plt.plot(figsize=(15, 6))
    
# Plot 1: PageRank scores comparison
rank_values = sorted(results_de.keys())
x = np.arange(len(page_labels))
width = 0.8 / len(rank_values)
    
for i, d in enumerate(rank_values):
    r, _ = results_de[d]
    offset = (i - len(rank_values)/2) * width + width/2
    plt.bar(x + offset, r, width, label=f'd = {d*100:.1f}%', alpha=0.8)
    
plt.xlabel('Page', fontsize=12, fontweight='bold')
plt.ylabel('PageRank Score', fontsize=12, fontweight='bold')
plt.title('Manipulation Effect Scale', fontsize=14, fontweight='bold')
plt.xticks(x, page_labels)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

print("\nPage |    25%     |    50%    |    85%")
print("-"*50)
for i, label in enumerate(page_labels):
    row_values = [results_de[k][0][i] for k in results_de.keys()]
    formatted_scores = " |".join(f"{val:10.6f}" for val in row_values)
    print(f"{label:<4} | {formatted_scores}" )